In [ ]:
# ============================================================
# PERSONALITY CLASSIFICATION - ANN BINARY CLASSIFICATION
# Target: Personality
# Classes: Introvert / Extrovert
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout


# ============================================================
# 2. LOAD DATASET
# ============================================================

file_path = r"C:\Users\Maruthi B\Downloads\personality_dataset_2000.csv"

df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows:")
print(df.head())


# ============================================================
# 3. BASIC DATA EXPLORATION
# ============================================================

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())


# ============================================================
# 4. REMOVE DUPLICATES
# ============================================================

df = df.drop_duplicates()

print("\nShape After Removing Duplicates:")
print(df.shape)


# ============================================================
# 5. HANDLE MISSING VALUES
# ============================================================

# Remove rows where Text or Personality is missing
df = df.dropna(
    subset=["Text", "Personality"]
)

print("\nShape After Removing Missing Values:")
print(df.shape)


# ============================================================
# 6. DISPLAY TARGET CLASSES
# ============================================================

print("\nPersonality Classes:")
print(df["Personality"].value_counts())


# ============================================================
# 7. DEFINE FEATURES AND TARGET
# ============================================================

X = df["Text"]

y = df["Personality"]


# ============================================================
# 8. CONVERT TARGET INTO NUMBERS
# ============================================================

# Introvert = 0
# Extrovert = 1

y = y.map({
    "Introvert": 0,
    "Extrovert": 1
})


print("\nEncoded Target:")
print(y.head())


# ============================================================
# 9. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y
)


print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)


# ============================================================
# 10. TF-IDF VECTORIZATION
# ============================================================

tfidf = TfidfVectorizer(

    max_features=5000,

    lowercase=True,

    stop_words="english"
)


# Learn vocabulary from training text
X_train_tfidf = tfidf.fit_transform(
    X_train
)


# Transform test text
X_test_tfidf = tfidf.transform(
    X_test
)


print("\nTF-IDF Training Shape:")
print(X_train_tfidf.shape)

print("\nTF-IDF Testing Shape:")
print(X_test_tfidf.shape)


# Convert sparse matrix to dense array
X_train_tfidf = X_train_tfidf.toarray()

X_test_tfidf = X_test_tfidf.toarray()


# ============================================================
# 11. BUILD ANN MODEL
# ============================================================

model = Sequential([

    # Input + First Hidden Layer
    Dense(
        128,
        activation="relu",
        input_shape=(X_train_tfidf.shape[1],)
    ),

    # Dropout
    Dropout(0.2),

    # Second Hidden Layer
    Dense(
        64,
        activation="relu"
    ),

    # Dropout
    Dropout(0.2),

    # Third Hidden Layer
    Dense(
        32,
        activation="relu"
    ),

    # Output Layer
    # 1 neuron because binary classification
    Dense(
        1,
        activation="sigmoid"
    )
])


# ============================================================
# 12. DISPLAY MODEL
# ============================================================

model.summary()


# ============================================================
# 13. COMPILE MODEL
# ============================================================

model.compile(

    optimizer="adam",

    loss="binary_crossentropy",

    metrics=["accuracy"]
)


# ============================================================
# 14. TRAIN ANN
# ============================================================

history = model.fit(

    X_train_tfidf,

    y_train,

    validation_split=0.20,

    epochs=30,

    batch_size=32,

    verbose=1
)


# ============================================================
# 15. PREDICTION
# ============================================================

y_probability = model.predict(
    X_test_tfidf
).flatten()


# Convert probability into class
# >= 0.5 → Extrovert
# < 0.5 → Introvert

y_pred = (
    y_probability >= 0.5
).astype(int)

# ============================================================
# 16. ACCURACY
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

# Convert accuracy to percentage
accuracy_percentage = accuracy * 100


print("\n================================")
print("ANN CLASSIFICATION RESULTS")
print("================================")

print(
    "Accuracy:",
    accuracy_percentage,
    "%"
)

# ============================================================
# 17. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Introvert",
            "Extrovert"
        ]
    )
)


# ============================================================
# 18. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


# ============================================================
# 19. CONFUSION MATRIX GRAPH
# ============================================================

plt.figure(figsize=(6, 5))

plt.imshow(cm)

plt.title(
    "Confusion Matrix - Personality Classification"
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.xticks(
    [0, 1],
    ["Introvert", "Extrovert"]
)

plt.yticks(
    [0, 1],
    ["Introvert", "Extrovert"]
)

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()

plt.show()


# ============================================================
# 20. TRAINING AND VALIDATION ACCURACY GRAPH
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.title(
    "ANN Training and Validation Accuracy"
)

plt.legend()

plt.show()


# ============================================================
# 21. TRAINING AND VALIDATION LOSS GRAPH
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Binary Crossentropy Loss")

plt.title(
    "ANN Training and Validation Loss"
)

plt.legend()

plt.show()


# ============================================================
# 22. EXTERNAL INPUT PREDICTION
# ============================================================

print("\n================================")
print("EXTERNAL INPUT PREDICTION")
print("================================")
# Enter a new sentence
new_text = [
   "I get so energized collaborating in a busy office, but I always put my headphones on and look down so nobody interrupts my thoughts while I'm coding."
]


# Convert new text using the SAME TF-IDF vectorizer
new_text_tfidf = tfidf.transform(
    new_text
).toarray()


# Predict probability
prediction_probability = model.predict(
    new_text_tfidf
)[0][0]


# Convert probability to class
if prediction_probability >= 0.5:

    prediction = "Extrovert"

else:

    prediction = "Introvert"


print(
    "\nInput Text:",
    new_text[0]
)

print(
    "Extrovert Probability:",
    prediction_probability
)

print(
    "Predicted Personality:",
    prediction
)

Dataset Shape: (2000, 2)

First 5 Rows:
                                             Text Personality
0                    I feel bored when alone #995   Extrovert
1  I prefer spending time alone reading books #70   Introvert
2          I enjoy solo hobbies like painting #90   Introvert
3                 I feel energized in crowds #408   Extrovert
4         I enjoy solo hobbies like painting #459   Introvert

Column Names:
['Text', 'Personality']

Data Types:
Text           str
Personality    str
dtype: object

Missing Values:
Text           0
Personality    0
dtype: int64

Duplicate Rows:
0

Shape After Removing Duplicates:
(2000, 2)

Shape After Removing Missing Values:
(2000, 2)

Personality Classes:
Personality
Extrovert    1000
Introvert    1000
Name: count, dtype: int64

Encoded Target:
0    1
1    0
2    0
3    1
4    0
Name: Personality, dtype: int64

Training Data: (1600,)
Testing Data: (400,)

TF-IDF Training Shape:
(1600, 1002)

TF-IDF Testing Shape:
(400, 1002)


C:\Users\Maruthi B\miniconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                     │ (None, 128)                 │         128,384 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_13 (Dense)                     │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_7 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_14 (Dense)                     │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_15 (Dense)                     │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 138,753 (542.00 KB)

 Trainable params: 138,753 (542.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.8234 - loss: 0.6432 - val_accuracy: 0.9969 - val_loss: 0.4888
Epoch 2/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9977 - loss: 0.1955 - val_accuracy: 1.0000 - val_loss: 0.0158
Epoch 3/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 1.0000 - loss: 0.0059 - val_accuracy: 1.0000 - val_loss: 0.0026
Epoch 4/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 1.0000 - loss: 0.0017 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 5/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 1.0000 - loss: 0.0010 - val_accuracy: 1.0000 - val_loss: 9.5624e-04
Epoch 6/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 1.0000 - loss: 6.6685e-04 - val_accuracy: 1.0000 - val_loss: 7.1114e-04
Epoch 7/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 1.0000 - loss: 4.4871e-04 - val_accuracy: 1.0000 - val_loss: 5.5739e-04
Epoch 8/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 1.0000 - loss: 3.6503e-04 - 